<a href="https://colab.research.google.com/github/saim873/flyrank-ml-internship-Saim/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Setup complete")

Setup complete


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [6]:
df = pd.read_csv("/content/content_refresh_anonymized.csv")

# Transparent action score using available historical fields
df["action_score"] = (
    (df["impressions_prev_30d"].fillna(0) * 0.40)
    + (df["impressions_last_30d"].fillna(0) * 0.20)
    + (df["search_volume"].fillna(0) * 0.20)
    + (df["days_since_last_update"].fillna(0) * 0.10)
    + (df["avg_position"].fillna(0) * 0.10)
)

def make_reason_code(row):
    reasons = []

    if row["days_since_last_update"] >= 90:
        reasons.append("STALE_CONTENT")

    if row["impressions_prev_30d"] >= 500:
        reasons.append("HISTORIC_VISIBILITY")

    if row["impressions_last_30d"] < row["impressions_prev_30d"]:
        reasons.append("RECENT_ACTIVITY_LOWER")

    if row["ctr"] < 1:
        reasons.append("WEAK_CTR")

    if row["avg_position"] > 10:
        reasons.append("WEAK_POSITION")

    if len(reasons) == 0:
        return "NO_STRONG_SIGNAL"

    if len(reasons) >= 3:
        return "MULTIPLE_SIGNALS"

    return "; ".join(reasons)

def assign_action(row):
    if row["action_score"] >= df["action_score"].quantile(0.90):
        return "Review and refresh first"
    elif row["action_score"] >= df["action_score"].quantile(0.70):
        return "Review when capacity allows"
    else:
        return "Monitor"

df["reason_code"] = df.apply(make_reason_code, axis=1)
df["recommended_action"] = df.apply(assign_action, axis=1)

ranked_queue = (
    df.sort_values(
        ["action_score", "impressions_prev_30d"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

ranked_queue["rank"] = np.arange(1, len(ranked_queue) + 1)

ranked_queue[
    [
        "rank",
        "content_id",
        "action_score",
        "recommended_action",
        "reason_code",
        "impressions_prev_30d",
        "impressions_last_30d",
        "days_since_last_update",
        "ctr",
        "avg_position"
    ]
].head(20)

,rank,content_id,action_score,recommended_action,reason_code,impressions_prev_30d,impressions_last_30d,days_since_last_update,ctr,avg_position
0,1,content_5fe46e04994d,112063.42,Review and refresh first,MULTIPLE_SIGNALS,218786,120791,104,0.14,4.2
1,2,content_aaef01a50def,106034.94,Review and refresh first,MULTIPLE_SIGNALS,177601,170559,22,0.25,5.4
2,3,content_2cb567c3c89b,103798.82,Review and refresh first,MULTIPLE_SIGNALS,160641,197677,48,0.10,22.2
3,4,content_9532f197bbc8,91570.00,Review and refresh first,MULTIPLE_SIGNALS,174235,109317,104,0.87,2.0
4,5,content_2c2606c5d176,86610.02,Review and refresh first,MULTIPLE_SIGNALS,164079,104248,104,0.53,4.2
5,6,content_2dba2b1f9536,83154.99,Review and refresh first,MULTIPLE_SIGNALS,137909,139891,104,0.21,27.9
6,7,content_8c19996aa890,82422.45,Review and refresh first,MULTIPLE_SIGNALS,161284,89463,20,0.15,2.5
7,8,content_1a9e894be2e2,80769.40,Review and refresh first,MULTIPLE_SIGNALS,147889,107986,22,0.23,4.0
8,9,content_8451fc6f034d,72838.23,Review and refresh first,HISTORIC_VISIBILITY; WEAK_CTR,97606,168958,20,0.03,2.3
9,10,content_44e481c8f55b,68152.54,Review and refresh first,MULTIPLE_SIGNALS,118137,104458,20,0.65,1.4


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## Intended use

This action playbook is designed to help content reviewers prioritize which content items should be reviewed first. The ranked queue is intended to support human decision-making by highlighting content that shows multiple measurable signals of declining performance or reduced freshness.

The recommendations are intended for planning content reviews and refresh activities. They should be used as decision-support rather than automatic actions.

---

## Limits

- The playbook does not prove that refreshing content will improve performance.
- The recommendations are based only on the available historical dataset.
- Results may not generalize to every client, topic, or future time period.
- Human review is required before any content changes are made.
- The ranked queue should be interpreted as directional evidence rather than a guaranteed outcome.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## Human review

Before taking any action, a reviewer should check:

- Whether the content is still accurate and up to date.
- Whether search intent has changed.
- Whether traffic changes are seasonal or temporary.
- Whether competitors have published stronger content.
- Whether technical SEO issues are affecting performance.
- Whether the recommendation is reasonable for the business objective.

---

## No-go list

The following actions should **not** be automated:

- Automatically updating or deleting content.
- Publishing AI-generated content without human review.
- Making business decisions based only on the model score.
- Assuming that refreshing content will always improve performance.
- Using the model outside the observed dataset without validation.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## Monitoring / Retrain Triggers

The action playbook should be reviewed and the model should be retrained when:

- Overall prediction accuracy decreases noticeably.
- New content types or clients are added.
- Search trends change significantly.
- Website structure or SEO strategy changes.
- The distribution of important features changes over time.
- The ranked recommendations no longer match human review outcomes.

The model is intended to provide directional decision-support and should be monitored regularly to maintain its usefulness.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [7]:
from pathlib import Path

# Create output folder
output_dir = Path("/content/work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Save ranked queue
ranked_queue.to_csv(
    output_dir / "content_action_playbook.csv",
    index=False
)

print("Export completed successfully.")
print("Saved file:")
print(output_dir / "content_action_playbook.csv")

Export completed successfully.
Saved file:
/content/work/outputs/content_action_playbook.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-check

- ✅ Every section above is filled with markdown and supporting code.
- ✅ The notebook runs from top to bottom without errors.
- ✅ No client names, URLs, or private queries are included.
- ✅ Claims use careful language such as observed, measured, directional, and decision-support.
- ✅ Ranked actions include reason codes and human-review guidance.
- ✅ Monitoring and retrain triggers are stated.
- ✅ The ranked queue is exported for the paper.
- ✅ The notebook will be committed under `work/notebooks/w07_action_playbook.ipynb`.